# 00. 실습 인프라 만들기 — IT 헬프데스크 에이전트를 AWS에 올리기

Demo 1~3이 함께 쓰는 실제 AWS 환경을 만듭니다. **강의 전에 한 번** 실행하세요(약 5~7분). 다 쓰고 나면 `99_cleanup.ipynb`로 지웁니다.

```
 직원(노트북) ──InvokeAgentRuntime──▶ AgentCore Runtime: 헬프데스크 에이전트 (Strands + Nova Lite)
                                         │  도구
                                         ├─ 사내 디렉터리 ............ Amazon Cognito 사용자 풀
                                         ├─ 본인 확인(OTP) ........... Secrets Manager(TOTP 시드) + DynamoDB
                                         ├─ KB 검색 ................... Bedrock Knowledge Base (S3 Vectors)
                                         ├─ 로그인 기록 ............... CloudWatch Logs
                                         └─ 티켓·변경·감사·승인 기록 .. DynamoDB
                                         │  관찰성 (ADOT 자동 계측)
                                         └─▶ CloudWatch: aws/spans(트레이스) · 런타임 로그 · 지표
 보안 경보 ─PutEvents─▶ EventBridge 규칙 ─▶ Lambda ─▶ AgentCore Runtime (범위 4: 사람 없이 시작)
```

| 단계 | 만드는 것 | 시간 |
|---|---|---|
| 1 | (계정 설정) X-Ray Transaction Search — 이미 켜져 있으면 바꾸지 않음 | 수 초 |
| 2 | DynamoDB 테이블, TOTP 시드, Cognito 디렉터리(사용자 7·그룹 6), 로그인 기록 800건 | 30초 |
| 3 | S3 + S3 Vectors + Bedrock Knowledge Base, KB 문서 6건 수집 | 1분 |
| 4 | AgentCore Runtime 배포 (의존성 패키징 → S3 → 직접 코드 배포) | 2~3분 |
| 5 | EventBridge 규칙 + Lambda (보안 경보 → 에이전트) | 30초 |
| 6 | 동작 확인: 에이전트 호출 · DynamoDB 기록 · CloudWatch 트레이스 | 2분 |

**필요한 것**: 관리자에 가까운 권한(IAM 역할 생성 포함), `pip install -r requirements.txt`, `uv`(의존성을 Linux arm64용으로 받기 위해, `pip install uv`). Docker는 필요 없습니다.

In [1]:
# %pip install -q -r requirements.txt   # 처음 한 번 (설치 후 커널 재시작)
import os, json, time, shutil
# os.environ["AWS_REGION"] = "us-east-1"   # 리전을 바꾸려면 주석을 풀고 커널 재시작 (없으면 AWS_DEFAULT_REGION → boto3 기본 리전, 가드레일 교차 리전 프로필: us-east-1 · us-east-2 · us-west-2)
os.environ.setdefault("LAB_SUFFIX", "demo")        # 같은 계정을 여럿이 쓰면 번호를 다르게
import boto3
import lab_infra as L

arn = boto3.client("sts").get_caller_identity()["Arn"]
print("자격 증명:", arn[:13] + "********" + arn[25:])
print("리전:", L.REGION, "| 이름 접두사:", L.P)
print("uv:", shutil.which("uv") or "❌ 없음 → pip install uv")

자격 증명: arn:aws:iam::********:user/bedrock-user25
리전: us-west-2 | 이름 접두사: ajou-aisec3x-demo
uv: C:\Users\moonj\.local\bin\uv.EXE


## 1. X-Ray Transaction Search (계정 설정)
에이전트 스팬을 CloudWatch Logs의 `aws/spans` 로그 그룹으로 보내는 계정 단위 설정입니다. AgentCore Observability와 CloudWatch GenAI Observability 대시보드가 이 설정을 씁니다. **이미 켜져 있으면 아무것도 바꾸지 않고**, 꺼져 있으면 켜고 그 사실을 `lab_config.json`에 기록합니다(정리할 때 되돌릴지 선택).

> 켜면 스팬 수집에 CloudWatch Logs 요금이 붙고, 같은 계정·리전의 다른 워크로드에도 적용됩니다.

In [2]:
ts = L.ensure_transaction_search()
print("변경 전:", ts["before"]["Destination"], ts["before"].get("Status"), "→", "켬" if ts["changed"] else "이미 켜져 있음 (변경 없음)")

변경 전: CloudWatchLogs ACTIVE → 이미 켜져 있음 (변경 없음)


## 2. 디렉터리 · 본인 확인 · 기록 저장소
- **Cognito 사용자 풀** = 사내 디렉터리. 계정 잠금은 `AdminDisableUser`, 해제는 `AdminEnableUser`, 비밀번호 초기화는 임시 비밀번호 설정(`Permanent=False`), 권한 그룹은 Cognito 그룹입니다.
- **TOTP 시드**(Secrets Manager): 직원 휴대폰의 인증 앱 역할. 노트북의 `H.otp("u-1024")`가 직원 휴대폰처럼 현재 6자리를 계산합니다. u-3001(재무팀장)은 등록하지 않았습니다 → 대리인은 OTP를 댈 수 없음.
- **DynamoDB** 단일 테이블: 본인 확인 결과, 실제 변경 기록, 티켓, 감사 기록, 승인 요청.

In [3]:
print("DynamoDB:", L.create_table())
print("TOTP 시드:", L.create_totp_secret())
print("Cognito 디렉터리:", L.create_directory())
print("로그인 기록:", L.create_login_events())
idp = boto3.client("cognito-idp", region_name=L.REGION)
pid = L.load()["user_pool_id"]
rows = []
for uid in L.USERS:
    u = idp.admin_get_user(UserPoolId=pid, Username=uid)
    a = {x["Name"]: x["Value"] for x in u["UserAttributes"]}
    g = [x["GroupName"] for x in idp.admin_list_groups_for_user(UserPoolId=pid, Username=uid)["Groups"]]
    rows.append(f"{uid} {a['name']} {a['custom:dept']:<4} 잠김={not u['Enabled']!s:<5} 그룹={g} OTP 등록={'예' if uid in L.TOTP_USERS else '아니오'}")
print("\n".join(rows))

DynamoDB: ajou-aisec3x-demo-helpdesk


TOTP 시드: ajou-aisec3x-demo/totp-seeds


Cognito 디렉터리: us-west-2_jOZ4FZxKo


로그인 기록: /ajou-aisec3x-demo/login-events


u-1024 김민수 영업팀  잠김=True  그룹=['Sales'] OTP 등록=예
u-2048 이서연 개발팀  잠김=False 그룹=['Dev'] OTP 등록=예
u-3001 박지훈 재무팀  잠김=False 그룹=['Finance-Approvers', 'Finance'] OTP 등록=아니오
u-3002 정다은 재무팀  잠김=False 그룹=['Finance'] OTP 등록=아니오
u-3003 한승우 재무팀  잠김=False 그룹=['Finance'] OTP 등록=아니오
u-3004 오지민 재무팀  잠김=False 그룹=['Finance'] OTP 등록=아니오
u-4096 최유진 인사팀  잠김=True  그룹=['HR'] OTP 등록=예


## 3. 사내 IT 지식 베이스 (Bedrock Knowledge Bases + S3 Vectors)
KB 문서 6건을 S3에 올리고, Titan Text Embeddings v2로 임베딩해 **S3 Vectors** 인덱스에 넣습니다. 에이전트의 `search_kb` 도구는 `bedrock-agent-runtime:Retrieve`를 호출합니다.

In [4]:
t = time.time()
kb = L.create_knowledge_base()
c = L.load()
print(f"Knowledge Base {kb} | 문서 {c.get('kb_docs_indexed')}건 수집 | {time.time() - t:.0f}초")
r = boto3.client("bedrock-agent-runtime", region_name=L.REGION).retrieve(
    knowledgeBaseId=kb, retrievalQuery={"text": "MFA 기기를 잃어버렸어요"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 1}})
print("검색 확인:", r["retrievalResults"][0]["content"]["text"][:80].replace("\n", " "))

Knowledge Base AQUQJZLNCG | 문서 6건 수집 | 1초


검색 확인: # KB-105 MFA 기기 분실  MFA 기기를 잃어버린 경우 신분증을 지참하고 IT 데스크(본관 3층)에서 대면으로 처리한다. 원격으로 MF


## 4. AgentCore Runtime에 에이전트 배포
`agent_runtime/helpdesk_agent.py`(진입점)와 `helpdesk_backend.py`(도구)를 배포합니다.
1. `uv`로 의존성을 **Linux arm64** 휠로 받아 패키지 폴더를 만듭니다(Windows·Mac에서도 동작).
2. zip으로 묶어 S3에 올립니다.
3. 실행 역할(IAM)을 만들고, 도구가 쓰는 권한만 붙입니다(Cognito 해당 풀, DynamoDB 해당 테이블, KB 검색, 해당 로그 그룹).
4. `create_agent_runtime`(직접 코드 배포, `opentelemetry-instrument`로 시작 → ADOT 자동 계측).

In [5]:
t = time.time()
dep = L.deploy_runtime(model_id=os.environ.get("DEMO_MODEL", "us.amazon.nova-lite-v1:0"))
print(f"{dep['status']} | 버전 {dep['version']} | 패키지 {dep['package_mb']} MB | {time.time() - t:.0f}초")
print("실행 역할에 붙인 도구 권한:")
pol = boto3.client("iam").get_role_policy(RoleName=L.load()["runtime_role"], PolicyName="helpdesk-tools")["PolicyDocument"]
for s in pol["Statement"]:
    print(f"  {s['Sid']:<10} {', '.join(a.split(':')[1] for a in (s['Action'] if isinstance(s['Action'], list) else [s['Action']]))[:90]}")

READY | 버전 4 | 패키지 39.8 MB | 64초
실행 역할에 붙인 도구 권한:


  Directory  AdminGetUser, AdminListGroupsForUser, AdminEnableUser, AdminDisableUser, AdminSetUserPassw
  Records    PutItem, GetItem, Query
  Totp       GetSecretValue
  Kb         Retrieve
  LoginLogs  FilterLogEvents
  Models     InvokeModel, InvokeModelWithResponseStream, ApplyGuardrail


## 5. 보안 경보 → EventBridge → Lambda → 에이전트 (Demo 2 범위 4)
`source = ajou.aisec3x.security`, `detail-type = Security Alert` 이벤트가 들어오면 Lambda가 **사람 없이** 에이전트를 호출합니다.

In [6]:
print(L.create_alert_pipeline())

{'function': 'ajou-aisec3x-demo-alert-responder', 'rule': 'ajou-aisec3x-demo-security-alert'}


## 6. 동작 확인
직원 u-1024가 OTP를 대고 계정 잠금 해제를 요청합니다. 에이전트 응답 → Cognito의 실제 상태 → DynamoDB 기록 → CloudWatch 트레이스를 차례로 확인합니다.

In [7]:
import importlib, helpdesk_common as H
importlib.reload(H)
H.check()
H.reset_directory(["u-1024"])
print("\n잠금 상태(전):", H.directory_state("u-1024")["locked"])
out = H.run_session({"prompt": f"제 계정 u-1024가 잠겼어요. OTP는 {H.otp('u-1024')}입니다. 풀어 주세요.",
                     "requester": "emp-1024", "scope": 3, "system": "secure"})
print("에이전트:", H.clean(out["text"])[:150])
print("잠금 상태(후):", H.directory_state("u-1024")["locked"], "| 도구:", " → ".join(out["calls"]))
print("DynamoDB 기록:", [f"{i['sk'].split('#')[0]}:{i.get('tool') or i.get('action') or ''}" for i in H.items(out["session_id"]) if not i["sk"].startswith("CALL")])
sp = H.wait_for_spans([out["session_id"]])
H.print_span_tree([s for s in sp if s.get("scope", {}).get("name", "").startswith("strands")])
H.reset_directory(["u-1024"])

자격 증명: arn:aws:iam::********7146:user/bedrock-user25
리전: us-west-2 | 모델: us.amazon.nova-lite-v1:0
AgentCore Runtime: ajou_aisec3x_demo_helpdesk-7PvUWFGnLH (버전 4, READY)
디렉터리: Cognito us-west-2_jOZ4FZxKo | KB: AQUQJZLNCG | 기록: DynamoDB ajou-aisec3x-demo-helpdesk



잠금 상태(전): True


에이전트: 안녕하세요, 사용자님. 귀하의 계정 잠금이 해제되었습니다. 다시 로그인해 보시기 바랍니다. 추가 도움이 필요하시면 언제든지 문의해 주세요.


잠금 상태(후): False | 도구: verify_identity → unlock_account


DynamoDB 기록: ['AUDIT:verify_identity', 'AUDIT:unlock_account', 'CHANGE:unlock_account', 'VERIFY:']


스팬 23개 수집 (세션 1/1, 2초 대기)
└ invoke_agent Strands Agents               2.94s  [strands.telemetry.tracer] 토큰 5181→315
   └ execute_event_loop_cycle                  1.02s  [strands.telemetry.tracer] 
      └ chat                                      0.92s  [strands.telemetry.tracer] 토큰 1580→108
      └ execute_tool verify_identity              0.09s  [strands.telemetry.tracer] 
   └ execute_event_loop_cycle                  0.93s  [strands.telemetry.tracer] 
      └ chat                                      0.76s  [strands.telemetry.tracer] 토큰 1731→90
      └ execute_tool unlock_account               0.16s  [strands.telemetry.tracer] 
   └ execute_event_loop_cycle                  0.98s  [strands.telemetry.tracer] 
      └ chat                                      0.98s  [strands.telemetry.tracer] 토큰 1870→117


여기까지 오류 없이 끝나면 Demo 1~3을 실행할 수 있습니다. 설정은 `lab_config.json`에 저장되어 있습니다.